In [1]:
import pandas as pd
import numpy as np

import pickle
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots




## LOAD INPUTS + WAVEFORM NUMERICS

In [4]:


mv_filtered_10min = pd.read_pickle('/Users/riccardoconci/Local_documents/Counterfactual_ICU/data/mimic_3_data/processed_data/mv_filtered_10min.pkl')
mv_filtered_10min.head()

,subject_id,hadm_id,item_id,input_name,input_class,start_time,end_time,rate,rate_uom,rate/weight,...,prev_rate,trigger,trigger_reason,action_cluster_id,action_cluster_size,action_cluster_rank,first,last,absolute_timestamp,wf_time_delta_s
0,124,138376,225828,03-IV Fluid Bolus,Bolus,2166-01-12 04:00:00+00:00,2166-01-12 04:01:00+00:00,NaN,NaN,NaN,...,0.649351,True,increase,13.0,1.0,1.0,2160-07-05 21:51:12.668000+00:00,2166-01-28 10:07:41+00:00,2166-01-12 04:00:01+00:00,1.0
1,124,138376,225828,02-Fluids (Crystalloids),Continuous IV,2166-01-12 08:59:00+00:00,2166-01-12 22:15:00+00:00,5.000000,mL/hour,0.064935,...,1.000000,True,decrease,14.0,1.0,1.0,2160-07-05 21:51:12.668000+00:00,2166-01-28 10:07:41+00:00,2166-01-12 08:59:01+00:00,1.0
2,124,138376,225158,01-Drips,Continuous Med,2166-01-12 18:00:00+00:00,2166-01-12 21:00:00+00:00,4.000000,mL/hour,0.051948,...,0.125035,False,decrease,NaN,NaN,NaN,2160-07-05 21:51:12.668000+00:00,2166-01-28 10:07:41+00:00,2166-01-12 18:00:01+00:00,1.0
3,124,138376,225158,01-Drips,Continuous Med,2166-01-12 21:00:00+00:00,2166-01-12 22:00:00+00:00,5.000000,mL/hour,0.064935,...,0.051948,False,,NaN,NaN,NaN,2160-07-05 21:51:12.668000+00:00,2166-01-28 10:07:41+00:00,2166-01-12 21:00:01+00:00,1.0
4,124,138376,225158,01-Drips,Continuous Med,2166-01-12 22:00:00+00:00,2166-01-13 00:00:00+00:00,8.006432,mL/hour,0.103980,...,0.064935,False,increase,NaN,NaN,NaN,2160-07-05 21:51:12.668000+00:00,2166-01-28 10:07:41+00:00,2166-01-12 22:00:01+00:00,1.0


In [3]:


df_clean = pd.read_pickle("/Users/riccardoconci/Local_documents/Counterfactual_ICU/data/mimic_3_data/processed_data/combined_waveforms/combined_waveforms.cleaned.pkl")
df_clean.head()

,hadm_id,record_name,absolute_timestamp,ABP MEAN,NBP MEAN,CVP,HR,RESP,record_start_time,record_end_time,icu_admission_time,time_seconds
0,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:00.056000+00:00,NaN,NaN,9.60,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,0
1,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:10.056000+00:00,NaN,NaN,9.75,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,10
2,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:20.056000+00:00,NaN,NaN,9.90,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,20
3,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:30.056000+00:00,NaN,NaN,10.05,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,30
4,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:40.056000+00:00,NaN,NaN,10.20,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,40


## CLEAN WAVEFORM + SMOOTH 

In [6]:
import numpy as np
import pandas as pd

def _smooth_1d_nanaware(arr: np.ndarray, neighbors: int,
                        keep_nan_center: bool = True,
                        min_valid: int = 1) -> np.ndarray:
    if neighbors <= 0 or arr.size == 0:
        return arr.astype(float, copy=True)
    win = 2*neighbors + 1
    s = pd.Series(arr, dtype="float64")
    sm = s.rolling(win, center=True, min_periods=min_valid).mean()
    if keep_nan_center:
        sm[s.isna()] = np.nan
    return sm.to_numpy()

def _clip_inplace(g: pd.DataFrame):
    if "ABP MEAN" in g:
        g["ABP MEAN"] = pd.to_numeric(g["ABP MEAN"], errors="coerce").clip(lower=30, upper=200)
    if "CVP" in g:
        g["CVP"] = pd.to_numeric(g["CVP"], errors="coerce").clip(lower=0, upper=40)
    return g

def _zero_center_cols(g: pd.DataFrame, cols, suffix="_zc"):
    for c in cols:
        if c in g:
            mu = g[c].mean(skipna=True)
            g[f"{c}{suffix}"] = g[c] - mu
    return g

def _zscore_cols(g: pd.DataFrame, cols, suffix="_zn"):
    for c in cols:
        if c in g:
            mu = g[c].mean(skipna=True)
            sd = g[c].std(skipna=True)
            g[f"{c}{suffix}"] = (g[c] - mu) / sd if (pd.notna(sd) and sd > 0) else np.nan
    return g

def _smooth_cols_multi(g: pd.DataFrame, cols, neighbors, source_suffixes, out_suffix):
    """
    Smooth multiple source variants (e.g., raw/zc/zn) in one go.
    For each c in cols and each src in source_suffixes, create c{src}{out_suffix}.
    """
    for c in cols:
        for src in source_suffixes:
            base = f"{c}{src}" if src else c
            if base in g:
                g[f"{base}{out_suffix}"] = _smooth_1d_nanaware(
                    pd.to_numeric(g[base], errors="coerce").to_numpy(),
                    neighbors=neighbors, keep_nan_center=True, min_valid=1
                )
    return g

def run_waveform_pipeline(
    df: pd.DataFrame,
    signals=("ABP MEAN","CVP","HR","RESP"),
    time_col="absolute_timestamp",
    group_cols=("hadm_id","record_name"),
    *,
    do_zero_center: bool = True,
    do_zscore: bool = True,
    smooth_neighbors: int = 120,
    smooth_variants=("zc","zn"),   # <- choose any of {"raw","zc","zn"}; e.g. ("zc","zn")
    out_suffix: str = "_ma120",
    flush_every_rows: int = 2_000_000,
):
    """
    Memory-friendly generator: per (hadm_id, record_name) it clips ABP/CVP,
    optionally creates _zc and/or _zn, then smooths any of the requested variants
    (e.g., zc and zn), yielding chunks so you can concat or write to disk.
    """
    need_time = time_col in df.columns
    out_frames, acc_rows = [], 0
    sort_keys = list(group_cols) + ([time_col] if need_time else [])
    df_sorted = df.sort_values(sort_keys)

    # map variant tokens to suffixes
    var2suf = {"raw": "", "zc": "_zc", "zn": "_zn"}
    src_suffixes = [var2suf[v] for v in smooth_variants]

    for _, g in df_sorted.groupby(list(group_cols), sort=False, dropna=False):
        # ensure numeric for signals
        for c in signals:
            if c in g:
                g[c] = pd.to_numeric(g[c], errors="coerce")

        # 1) clip only ABP MEAN and CVP
        g = _clip_inplace(g)

        # 2) normalization variants (per group)
        if do_zero_center:
            g = _zero_center_cols(g, signals, suffix="_zc")
        if do_zscore:
            g = _zscore_cols(g, signals, suffix="_zn")

        # 3) smoothing for any requested variants
        if smooth_neighbors and smooth_neighbors > 0 and src_suffixes:
            g = _smooth_cols_multi(g, signals, neighbors=smooth_neighbors,
                                   source_suffixes=src_suffixes, out_suffix=out_suffix)

        out_frames.append(g)
        acc_rows += len(g)
        if acc_rows >= flush_every_rows:
            yield pd.concat(out_frames, ignore_index=True)
            out_frames.clear()
            acc_rows = 0

    if out_frames:
        yield pd.concat(out_frames, ignore_index=True)

In [ ]:
chunks = []
for chunk in run_waveform_pipeline(
        df_clean,
        signals=("ABP MEAN","CVP","HR","RESP"),
        do_zero_center=True,
        do_zscore=True,
        smooth_neighbors=120,
        smooth_variants=("zc","zn"),  # <— smooth both zero-centered and z-scored
        out_suffix="_ma120",
        flush_every_rows=1_000_000):
    chunks.append(chunk)

df_final = pd.concat(chunks, ignore_index=True)

## PLOT INPUTS + WAVEFORMS FOR ANY HADM_ID IN MV

In [ ]:
def plot_waveforms_with_mv_inputs(
    combined_waveform_df: pd.DataFrame,
    input_mv_triggers: pd.DataFrame,
    hadm_id=None,
    start=None,
    end=None,
    resample=None,
    signals=("ABP MEAN","NBP MEAN","CVP","HR","RESP"),
    line_width=1.3,
    trigger_only=True,
    time_col="start_time",
    cluster_col="action_cluster",
    signal_variant="raw",              # 'raw' | 'zc' | 'zc_sm' | 'custom'
    custom_suffix=None,                # e.g. "_ma15", "_sm2", "_g"
):
    wf = combined_waveform_df.copy()
    mv = input_mv_triggers.copy()

    # --- suffix selection ---
    suffix_map = {"raw": "", "zc": "_zc", "zc_sm": "_zc_sm"}
    if signal_variant == "custom":
        if not custom_suffix:
            raise ValueError("When signal_variant='custom', provide custom_suffix (e.g., '_ma15').")
        wanted_suffix = custom_suffix
    else:
        if signal_variant not in suffix_map:
            raise ValueError("signal_variant must be one of {'raw','zc','zc_sm','custom'}")
        wanted_suffix = suffix_map[signal_variant]

    # resolve columns to plot
    plot_cols, titles = [], []
    for base in signals:
        candidate = base + wanted_suffix if wanted_suffix else base
        if candidate in wf.columns:
            plot_cols.append(candidate)
            tag = (signal_variant if signal_variant != "raw" else None)
            if signal_variant == "custom": tag = custom_suffix
            titles.append(base if tag is None else f"{base} ({tag})")
        elif base in wf.columns:
            plot_cols.append(base)   # fallback
            titles.append(f"{base} (fallback)")
        else:
            plot_cols.append(None)
            titles.append(f"{base} (missing)")

    # --- timestamps + filtering unchanged ---
    wf["absolute_timestamp"] = pd.to_datetime(wf["absolute_timestamp"], errors="coerce", utc=True)
    mv[time_col]             = pd.to_datetime(mv[time_col], errors="coerce", utc=True)
    if hadm_id is not None:
        if "hadm_id" in wf.columns: wf = wf[wf["hadm_id"] == hadm_id]
        if "hadm_id" in mv.columns: mv = mv[mv["hadm_id"] == hadm_id]
    if trigger_only and "trigger" in mv.columns:
        mv = mv[mv["trigger"] == True]

    # auto window
    if start is None or end is None:
        mins = [t for t in [wf["absolute_timestamp"].min(), mv[time_col].min()] if pd.notna(t)]
        maxs = [t for t in [wf["absolute_timestamp"].max(), mv[time_col].max()] if pd.notna(t)]
        if not mins or not maxs:
            raise ValueError("No timestamps found after filtering; check hadm_id or inputs.")
        start = min(mins) if start is None else pd.to_datetime(start, utc=True)
        end   = max(maxs) if end   is None else pd.to_datetime(end,   utc=True)
    else:
        start = pd.to_datetime(start, utc=True); end = pd.to_datetime(end, utc=True)

    wf = wf[(wf["absolute_timestamp"] >= start) & (wf["absolute_timestamp"] <= end)]
    mv = mv[(mv[time_col] >= start) & (mv[time_col] <= end)]
    wf = wf.sort_values("absolute_timestamp"); mv = mv.sort_values(time_col)

    if resample:
        keep_cols = ["absolute_timestamp", *[c for c in plot_cols if c]]
        wf = (wf[keep_cols].set_index("absolute_timestamp").resample(resample).mean().reset_index())

    # --- plotting unchanged below ---
    nrows = len(signals)
    fig = make_subplots(rows=nrows, cols=1, shared_xaxes=True, vertical_spacing=0.02, subplot_titles=titles)
    clusters = mv[cluster_col].dropna().astype(str).unique() if cluster_col in mv.columns else []
    palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Plotly
    color_map = {lab: palette[i % len(palette)] for i, lab in enumerate(sorted(clusters))}

    y_limits = {}
    for r, (base, col) in enumerate(zip(signals, plot_cols), start=1):
        if not col or col not in wf.columns:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", showlegend=False), row=r, col=1)
            y_limits[base] = (0, 1); continue
        y = pd.to_numeric(wf[col], errors="coerce")
        fig.add_trace(go.Scattergl(x=wf["absolute_timestamp"], y=y, mode="lines", name=col,
                                   line=dict(width=line_width), showlegend=False), row=r, col=1)
        finite = np.isfinite(y.to_numpy())
        if finite.any():
            ymin, ymax = np.nanmin(y), np.nanmax(y)
            y_limits[base] = (ymin - 0.05*(ymax-ymin) if ymin!=ymax else float(ymin)-1,
                              ymax + 0.05*(ymax-ymin) if ymin!=ymax else float(ymax)+1)
        else:
            y_limits[base] = (0, 1)

    for r, base in enumerate(signals, start=1):
        ymin, ymax = y_limits[base]
        for lab in sorted(clusters):
            times = mv.loc[mv[cluster_col].astype(str) == lab, time_col]
            if times.empty: continue
            first = True
            for t in times:
                fig.add_trace(go.Scatter(x=[t, t], y=[ymin, ymax], mode="lines",
                                         line=dict(color=color_map[lab], width=1.3, dash="dot"),
                                         name=str(lab), legendgroup=str(lab),
                                         showlegend=(r == 1 and first),
                                         hoverinfo="text", text=[f"{lab}<br>{t}", f"{lab}<br>{t}"]), row=r, col=1)
                first = False

    fig.update_layout(height=220*nrows, hovermode="x unified",
                      margin=dict(t=40,b=40,l=50,r=10), legend_title_text="MV action clusters")
    for r, base in enumerate(signals, start=1):
        fig.update_yaxes(range=list(y_limits[base]), row=r, col=1, title_text=base)
    fig.update_xaxes(title_text="Time", range=[start, end])
    return fig

In [ ]:
fig = plot_waveforms_with_mv_inputs(df_final, mv_filtered_10min,
    hadm_id=192494, cluster_col="action_cluster_id",
    signal_variant="raw")
fig.show()


In [ ]:
# plot the smoothed columns by asking the plotter for a custom suffix:
fig = plot_waveforms_with_mv_inputs(
    df_sm,
    mv_filtered_10min,
    hadm_id=101662,
    cluster_col="action_cluster_id",
    signal_variant="custom",
    custom_suffix="_ma15"               # uses e.g. 'ABP MEAN_ma15'
)
fig.show()